# **Data**

The data is  from Aquino et al (2019)

In [57]:
import subprocess
import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath("../utils"))
from preprocessing_tools import *
from calculate_g2p_accuracy import calculate_accuracy

# Data pre-processing


In [58]:
current_dir = Path().resolve()

scripts_dir = current_dir.parent / "scripts"
sh_files = [f for f in os.listdir(scripts_dir) if f.endswith(".sh")]

for file in sh_files:

    file_dir = scripts_dir / file

    try:
        # Read the file in binary mode
        with open(file_dir, "rb") as f:
            content = f.read()
        
        # Replace Windows line endings (\r\n) with Linux line endings (\n)
        content = content.replace(b"\r\n", b"\n")
        
        # Write the corrected content back
        with open(file_dir, "wb") as f:
            f.write(content)
            
        print("Success! Converted line endings to LF.")
    except FileNotFoundError:
        print(f"Could not find the file at {file_dir}")

Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.


In [59]:
paths_dir = current_dir / "paths"
paths_dir.mkdir(parents=True, exist_ok=True)

getsh_files = [f for f in os.listdir(scripts_dir) if f.endswith(".sh") and f.startswith("get-")]

for file in getsh_files[:-1]:
    try:
        # 3. Run the script
        result = subprocess.run(
            ["bash", file], 
            cwd=scripts_dir,                    
            check=True,
            capture_output=True,
            text=True
        )
        print(f"You should now see {file} inside the {paths_dir} folder.")
        
    except subprocess.CalledProcessError as e:
        print(f"Script failed with exit code {e.returncode}")
        print(f"Error details:\n{e.stderr}")
    except FileNotFoundError:
        print(f"Could not find the scripts directory at: {scripts_dir}")

You should now see get-csv-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-testref-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-log-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-log-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.


In [60]:
txt_files = [f for f in os.listdir(paths_dir) if f.endswith(".txt")]

train_files = []
test_files = []
testref_files = []

for file in txt_files:
    file_path = paths_dir / file
    if "train" in file:
        train_files.append(file_path)
    elif "testref" in file:
        testref_files.append(file_path)
    else:
        test_files.append(file_path)


## **Building the train, validation and test sets**

We will begin with creating the **traininig** dictionary, and its data is sourced from `_small/train`.

In [61]:
dicts_dir = current_dir / "dicts"

create_dict(train_files[1], train_files[0], dicts_dir / "train.dict")

../data/_small/train/TGL/0812.110816.021250.csv
../data/_small/train/TGL/2359.110818.085852.csv


In [ ]:
wlist_dir = current_dir / "wlist"

create_wlist(test_files[1], wlist_dir / "test.wlist")

# **Baseline Training**

The baseline model will be from Aquino et al (2019), specifically their monolingual tagalog G2P.


In [63]:
dicts_dir = current_dir / "dicts"

create_dict(train_files[1], train_files[0], dicts_dir / "train.dict")

../data/_small/train/TGL/0812.110816.021250.csv
../data/_small/train/TGL/2359.110818.085852.csv


In [64]:
result = subprocess.run(
    ["phonetisaurus-train", "--lexicon", dicts_dir / "train.dict", "--seq2_del"],
    capture_output=True,
    text=True
)

print(result.stderr)

INFO:phonetisaurus-train:2026-03-30 07:41:21:  Checking command configuration...
INFO:phonetisaurus-train:2026-03-30 07:41:21:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-03-30 07:41:21:  Aligning lexicon...
INFO:phonetisaurus-train:2026-03-30 07:41:23:  Training joint ngram model...
INFO:phonetisaurus-train:2026-03-30 07:41:23:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-03-30 07:41:23:  G2P training succeeded: train/model.fst



In [65]:
model_dir = current_dir / "train"

output_txt = current_dir / "output" / "baseline_output.txt"

with open(output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model", str(model_dir / "model.fst"),
            "--word_list", str(wlist_dir / "test.wlist"),
        ],
        stdout=f,
        text=True
    )



In [66]:
create_trxn(test_files[1], output_txt)

calculate_accuracy(test_files[0], testref_files[0])

../data/_small/test/TGL/0156.110816.035620.csv
../data/_small/test/TGL/0156.110816.035620.csv
../data/_small/testref/TGL/0156.110816.035620.csv
total phones in reference: 11089
total edit distance:       651
phone error rate:          0.05870682658490396


# **Hyperparameter Optimization**

Grid search over 6 Phonetisaurus hyperparameters using 5-fold cross-validation on the training dictionary, following the methodology of Section 4.2.4. Covers 160 configurations total.

In [ ]:
import itertools
import tempfile
import random
import pandas as pd
from pathlib import Path

# Load training dictionary
train_dict_path = dicts_dir / "train.dict"
with open(train_dict_path) as f:
    train_entries = [line.strip() for line in f if line.strip()]

print(f"Total training entries: {len(train_entries)}")

# Hyperparameter grid (Table 4.1)
param_grid = {
    "ngram_order": [3, 4, 5, 6, 7],
    "seq1_max":    [1, 2, 3, 4],
    "seq2_max":    [1, 2],
    "seq1_del":    [True, False],
    "seq2_del":    [True, False],
    "grow":        [True, False],
}

keys = list(param_grid.keys())
configs = list(itertools.product(*param_grid.values()))
print(f"Total configurations: {len(configs)}")

# 5-fold word-level split (fixed seed for reproducibility)
random.seed(42)
shuffled = train_entries.copy()
random.shuffle(shuffled)

N_FOLDS = 5
fold_size = len(shuffled) // N_FOLDS
folds = [shuffled[i * fold_size:(i + 1) * fold_size] for i in range(N_FOLDS)]
folds[-1] += shuffled[N_FOLDS * fold_size:]  # remainder into last fold
print(f"Fold sizes: {[len(f) for f in folds]}")

In [ ]:
def compute_per_cv(held_out_lines, apply_stdout):
    """
    Compute PER for a CV fold.
    held_out_lines : list of 'word p1 p2 ...' strings (gold standard)
    apply_stdout   : stdout from phonetisaurus-apply (word\tphonemes per line)
    """
    # Parse phonetisaurus-apply predictions
    pred = {}
    for line in apply_stdout.strip().split("\n"):
        if not line.strip():
            continue
        parts = line.split("\t")
        if len(parts) >= 2:
            pred[parts[0]] = parts[1].split()

    total_ref = 0
    total_edit = 0
    for line in held_out_lines:
        parts = line.split()
        if len(parts) < 2:
            continue
        word, gold = parts[0], parts[1:]
        hypothesis = pred.get(word, [])
        total_ref += len(gold)
        total_edit += align_trxn(gold, hypothesis)[2]

    return total_edit / total_ref if total_ref > 0 else 1.0

In [ ]:
results_path = current_dir / "output" / "hyperparam_results.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)

rows = []

for config_idx, config_vals in enumerate(configs):
    config = dict(zip(keys, config_vals))
    fold_pers = []

    for k in range(N_FOLDS):
        held_out  = folds[k]
        train_fold = [e for i, fold in enumerate(folds) if i != k for e in fold]

        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            # Write fold training dict
            fold_dict = tmpdir / "fold.dict"
            fold_dict.write_text("\n".join(train_fold))

            # Write held-out word list (deduplicated, preserving order)
            seen = set()
            wlist_lines = []
            for line in held_out:
                parts = line.split()
                if parts and parts[0] not in seen:
                    seen.add(parts[0])
                    wlist_lines.append(parts[0])
            held_wlist = tmpdir / "held.wlist"
            held_wlist.write_text("\n".join(wlist_lines))

            # Build training command
            cmd = [
                "phonetisaurus-train",
                "--lexicon",     str(fold_dict),
                "--ngram_order", str(config["ngram_order"]),
                "--seq1_max",    str(config["seq1_max"]),
                "--seq2_max",    str(config["seq2_max"]),
            ]
            if config["seq1_del"]: cmd.append("--seq1_del")
            if config["seq2_del"]: cmd.append("--seq2_del")
            if config["grow"]:     cmd.append("--grow")

            train_result = subprocess.run(
                cmd, cwd=tmpdir, capture_output=True, text=True
            )

            if train_result.returncode != 0:
                fold_pers.append(1.0)  # penalise failed configs
                continue

            # Apply model to held-out words
            apply_result = subprocess.run(
                [
                    "phonetisaurus-apply",
                    "--model",     str(tmpdir / "train" / "model.fst"),
                    "--word_list", str(held_wlist),
                ],
                capture_output=True, text=True, cwd=tmpdir
            )

            fold_pers.append(compute_per_cv(held_out, apply_result.stdout))

    mean_per = sum(fold_pers) / len(fold_pers)
    rows.append({**config, "mean_per": mean_per})

    # Save incrementally so progress isn't lost
    pd.DataFrame(rows).to_csv(results_path, index=False)

    if (config_idx + 1) % 10 == 0:
        best_so_far = min(r["mean_per"] for r in rows)
        print(f"[{config_idx + 1}/{len(configs)}] best so far: {best_so_far:.4f}")

print("Grid search complete.")

In [ ]:
results_df = pd.read_csv(results_path).sort_values("mean_per").reset_index(drop=True)

print("Top 10 configurations:")
print(results_df.head(10).to_string(index=False))

best = results_df.iloc[0]
print(f"\nBest config:")
for k, v in best.items():
    print(f"  {k}: {v}")

## **Re-train on Full Dictionary with Best Config**

In [ ]:
best_cmd = [
    "phonetisaurus-train",
    "--lexicon",     str(dicts_dir / "train.dict"),
    "--ngram_order", str(int(best["ngram_order"])),
    "--seq1_max",    str(int(best["seq1_max"])),
    "--seq2_max",    str(int(best["seq2_max"])),
]
if best["seq1_del"]: best_cmd.append("--seq1_del")
if best["seq2_del"]: best_cmd.append("--seq2_del")
if best["grow"]:     best_cmd.append("--grow")

print("Training command:", " ".join(best_cmd))

result = subprocess.run(best_cmd, capture_output=True, text=True)
print(result.stderr)

## **Evaluate Tuned Model on Test Set**

In [ ]:
tuned_output_txt = current_dir / "output" / "tuned_output.txt"

with open(tuned_output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model",     str(model_dir / "model.fst"),
            "--word_list", str(wlist_dir / "test.wlist"),
        ],
        stdout=f, text=True
    )

create_trxn(test_files[1], tuned_output_txt)

print("Tuned model:")
calculate_accuracy(test_files[0], testref_files[0])